# Import Libs

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)
import numpy as np, pandas as pd, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#  Import & Preprocess Data

In [ ]:
# -------------------------------------------------------------
# 1.  Prepare labels (same mapping as before)
# -------------------------------------------------------------
# @title Load the dataset
url = 'https://raw.githubusercontent.com/pinchunc/NMA_DL_SentimentAnalysis/main/data/hippoCorpusV2.csv'
df_dataset = pd.read_csv(url)

# @title filter for recalled memtype
df_dataset = df_dataset[df_dataset['memType'] == 'recalled']

df_bt = df_dataset.copy()
map_  = {1:"Low", 2:"-", 3:"High", 4:"High", 5:"High"}
df_bt["stressful"] = df_bt["stressful"].map(map_)
df_bt = df_bt[df_bt["stressful"] != "-"]

docs   = df_bt["story"].tolist()
labels = df_bt["stressful"].tolist()
lbl2id = {lbl: i for i, lbl in enumerate(sorted(set(labels)))}
y      = np.array([lbl2id[l] for l in labels])

# -------------------------------------------------------------
# 2.  Train / test split
# -------------------------------------------------------------
docs_train, docs_test, y_train, y_test = train_test_split(
    docs, y, test_size=0.25, random_state=42, stratify=y)

# Embedding

In [ ]:
# -------------------------------------------------------------
# 3.  Encode to embeddings (GPU if available)
# -------------------------------------------------------------
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
X_train = embed_model.encode(docs_train, batch_size=64, device=DEVICE, show_progress_bar=True)
X_test  = embed_model.encode(docs_test,  batch_size=64, device=DEVICE, show_progress_bar=True)


# Classify with a neural net

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

train_dl = DataLoader(TensorDataset(X_train_t, y_train_t),
                      batch_size=64, shuffle=True)
test_dl  = DataLoader(TensorDataset(X_test_t, y_test_t),
                      batch_size=256, shuffle=False)

class TinyMLP(nn.Module):
    def __init__(self, in_dim, hidden_dims=[128, 32], n_classes=2, p_drop=0.2):
        super().__init__()
        layers = []
        dims = [in_dim] + hidden_dims
        for i in range(len(dims)-1):
            layers += [nn.Linear(dims[i], dims[i+1]), # batch_first=True argument for 
                       nn.ReLU(),
                       nn.Dropout(p_drop)]
        # layers += [nn.GRU(dims[-1], dims[-1], batch_first=True), nn.ReLU()]
        layers += [nn.Linear(dims[-1], n_classes)]   # final logits
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # GRU outputs a tuple (output, hidden), take the output
        for layer in self.net:
            if isinstance(layer, nn.GRU):
                x, _ = layer(x)
            else:
                x = layer(x)
        return x            # raw logits (CrossEntropyLoss will apply Softmax)

In [ ]:
in_dim = X_train.shape[1]   # 384 for all‑MiniLM‑L6‑v2
model = TinyMLP(in_dim).to(DEVICE)
print(model)

In [ ]:
from sklearn.utils import compute_class_weight
from transformers import get_linear_schedule_with_warmup  # NEW

classes = np.unique(y_train)
weights_arr  = compute_class_weight("balanced",
                              classes=classes,
                              y=y_train)
class_w  = torch.tensor(weights_arr, dtype=torch.float32).to(DEVICE)
class_w_dict   = dict(zip(classes, weights_arr))


criterion = nn.CrossEntropyLoss(weight=class_w)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

# -------- scheduler --------
EPOCHS      = 200                   # keep, or lower if you like
total_steps = len(train_dl) * EPOCHS
warmup_pct  = 0.05                   # 5 % of steps used for warm‑up
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(warmup_pct * total_steps),
    num_training_steps=total_steps
)

In [ ]:
def train_epoch(dl, scheduler):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        scheduler.step()

        loss_sum += loss.item() * len(xb)
        preds     = logits.argmax(1)
        correct  += (preds == yb).sum().item()
        total    += len(xb)
    return loss_sum/total, correct/total

def eval_epoch(dl):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss   = criterion(logits, yb)

            loss_sum += loss.item()*len(xb)
            preds     = logits.argmax(1)
            correct  += (preds == yb).sum().item()
            total    += len(xb)
    return loss_sum/total, correct/total

In [ ]:
import matplotlib.pyplot as plt
# -------------------------------------------
# 1.  Initialise history containers
# -------------------------------------------
tr_loss_hist, val_loss_hist = [], []
tr_acc_hist,  val_acc_hist  = [], []

patience     = 10        # epochs with NO improvement allowed
best_val     = float("inf")
best_state   = None
wait         = 0
# -------------------------------------------
# 2.  Training loop (unchanged except for logging)
# -------------------------------------------
EPOCHS = 200
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(train_dl, scheduler)
    vl_loss, vl_acc = eval_epoch(test_dl)

    tr_loss_hist.append(tr_loss)
    val_loss_hist.append(vl_loss)
    tr_acc_hist.append(tr_acc)
    val_acc_hist.append(vl_acc)

    print(f"Epoch {epoch:02d} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {vl_loss:.4f} acc {vl_acc:.3f}")

    # -------- early stopping --------
    if vl_loss < best_val - 1e-3:   # significant improvement
        best_val   = vl_loss
        best_state = model.state_dict()
        wait       = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"\nEarly stopping at epoch {epoch} "
                  f"(no val‑loss improvement for {patience} epochs).")
            break
            
if best_state is not None:
    model.load_state_dict(best_state)

# -------------------------------------------
# 3.  Plot loss
# -------------------------------------------
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(tr_loss_hist) + 1), tr_loss_hist, label="Train loss")
plt.plot(range(1, len(val_loss_hist) + 1), val_loss_hist, label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Cross‑entropy loss")
plt.title("Training vs. validation loss")
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------------------
# 4.  Plot accuracy
# -------------------------------------------
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(tr_acc_hist) + 1), tr_acc_hist, label="Train accuracy")
plt.plot(range(1, len(val_acc_hist) + 1), val_acc_hist, label="Val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs. validation accuracy")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model.eval()
y_pred = []
with torch.no_grad():
    for xb,_ in test_dl:
        logits = model(xb.to(DEVICE))
        y_pred.append(logits.argmax(1).cpu())
y_pred = torch.cat(y_pred).numpy()

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred,
                            target_names=[k for k,_ in sorted(lbl2id.items())]))
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))


# Classify with ML

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.svm import SVC 

from sklearn.preprocessing import FunctionTransformer, StandardScaler

pipe_svm = Pipeline([
    ("pca", PCA(n_components=128, random_state=42)),
    ("clf", SVC(kernel="rbf",
                class_weight=class_w_dict,
                probability=True))
])
pipe_svm.fit(X_train, y_train)
y_pred = pipe_svm.predict(X_test)
print("LR + PCA accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred,
                            target_names=[k for k,_ in sorted(lbl2id.items(), key=lambda x:x[1])]))
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

# Performance based only in the word frequency - Semantics not allowed!

In [ ]:
import re
import spacy
import nltk, warnings
from nltk import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Download NLTK data the first time
nltk.download("stopwords")
nltk.download('punkt_tab')

# SpaCy English pipeline (for POS tags → better lemmatisation)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

stop_words = set(stopwords.words("english"))
lemmatizer  = WordNetLemmatizer()

# @title NLP preprocessing functions
def preprocess(text: str):
    """Tokenise, drop punctuation/stop-words, return clean tokens."""
    # Lower-case + keep only alphabetic characters (`re` faster than SpaCy for this bit)
    tokens = word_tokenize(text.lower())
    tokens = [re.sub(r"[^a-z]", "", tok) for tok in tokens]  # strip digits & punctuation
    tokens = [tok for tok in tokens if tok not in stop_words]
    return tokens

def lemmatise_tokens(tokens):
    doc = nlp(" ".join(tokens))          # faster batching than token-by-token
    return [tok.lemma_ for tok in doc if tok.lemma_ != "-PRON-"]  # remove SpaCy pronoun tag

##################
def text_to_lemmas(text: str):
    """Full clean‑up → single space‑joined lemma string."""
    toks   = preprocess(text)
    lemmas = lemmatise_tokens(toks)
    return " ".join(lemmas)

# -------------------------------------------------------------
# 1.  Preprocess train / test splits *separately*
# -------------------------------------------------------------
docs_train_clean = [text_to_lemmas(t) for t in docs_train]
docs_test_clean  = [text_to_lemmas(t) for t in docs_test]

# -------------------------------------------------------------
# 2.  Fit TF‑IDF *only on training data* (avoids data leakage)
# -------------------------------------------------------------
tfidf = TfidfVectorizer(min_df=2)          # min_df=2 = ignore 1‑offs
X_train = tfidf.fit_transform(docs_train_clean)
X_test  = tfidf.transform(docs_test_clean)  # use the same vocab
##################

# 5.  Build the PCA → SVM pipeline
pipe_svm = Pipeline([
    # ✳️  PCA needs dense input.  This transformer converts the sparse TF‑IDF
    #    matrix to a dense NumPy array.  If memory is a concern, replace the
    #    two lines below with ('svd', TruncatedSVD(n_components=128)) *without*
    #    the FunctionTransformer or StandardScaler.
    ("to_dense", FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ("pca", PCA(n_components=128, random_state=42)),
    # SVM usually benefits from scaling when inputs are dense
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf",
                class_weight=class_w_dict,
                probability=True,
                random_state=42))
])

# 6.  Fit & evaluate
pipe_svm.fit(X_train, y_train)
y_pred = pipe_svm.predict(X_test)

print("PCA + SVM accuracy :", accuracy_score(y_test, y_pred))
print("Macro‑F1           :", f1_score(y_test, y_pred, average="macro"))
print("\nClassification Report")
print(classification_report(y_test, y_pred,
                            target_names=[k for k,_ in sorted(lbl2id.items(), key=lambda x: x[1])]))
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))